# Nordvik – analiza sprzedaży i churn (przykładowy notebook)

Cel: pokazać typowy przepływ data science w Nordvik:
1. Dane (syntetyczne, wygenerowane tu – bez zależności od repo)
2. Eksploracja i jakość danych
3. Analiza sprzedaży i kategorii
4. Model churn (klasyfikacja) z walidacją bez wycieku danych
5. Wnioski biznesowe

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Dane syntetyczne

Symulujemy klientów, zamówienia i pozycje – te same nazwy co w `nordvik_sample_queries.sql`.

In [ ]:
N_CUSTOMERS = 5_000
END = pd.Timestamp("2026-11-08")
START = END - pd.Timedelta(days=365)

customers = pd.DataFrame({
    "customer_id": np.arange(1, N_CUSTOMERS + 1),
    "city": RNG.choice(["Oslo", "Bergen", "Trondheim", "Stavanger", "Tromsø"], N_CUSTOMERS, p=[.35, .2, .2, .15, .1]),
    "segment": RNG.choice(["retail", "premium", "b2b"], N_CUSTOMERS, p=[.7, .2, .1]),
    "created_at": START - pd.to_timedelta(RNG.integers(0, 900, N_CUSTOMERS), unit="D"),
})

# Każdy klient ma wewnętrzną "skłonność" do zakupów; część klientów odpada (churn)
customers["rate"] = RNG.gamma(2.0, 0.6, N_CUSTOMERS)
customers["churn_day"] = END - pd.to_timedelta(RNG.integers(0, 365, N_CUSTOMERS), unit="D")
customers["will_churn"] = RNG.random(N_CUSTOMERS) < 0.25
customers.loc[~customers["will_churn"], "churn_day"] = END + pd.Timedelta(days=1)

orders = []
for cid, rate, churn_day in customers[["customer_id", "rate", "churn_day"]].itertuples(index=False):
    n = RNG.poisson(rate * 6)
    days = RNG.integers(0, 365, n)
    for d in days:
        ts = START + pd.Timedelta(days=int(d), hours=int(RNG.integers(8, 22)))
        if ts < churn_day:
            orders.append((cid, ts, RNG.choice(["web", "app", "store"], p=[.3, .3, .4])))
orders = pd.DataFrame(orders, columns=["customer_id", "ordered_at", "channel"])
orders.insert(0, "order_id", np.arange(1, len(orders) + 1))
orders["status"] = np.where(RNG.random(len(orders)) < 0.04, "cancelled", "completed")

products = pd.DataFrame({
    "product_id": range(1, 9),
    "name": ["Kawa ziarnista", "Ekspres", "Czajnik", "Ręcznik", "Pościel", "Lampka", "Plecak", "Kurtka"],
    "category": ["Kuchnia", "AGD", "AGD", "Dom", "Dom", "Dom", "Moda", "Moda"],
    "unit_cost": [18, 220, 60, 12, 45, 30, 70, 150],
    "unit_price": [39, 499, 129, 29, 99, 79, 149, 349],
})

items = []
for oid in orders["order_id"]:
    for pid in RNG.choice(products["product_id"], RNG.integers(1, 4), replace=False):
        items.append((oid, pid, int(RNG.integers(1, 3)), float(RNG.choice([0, 0, 0, 10, 20]))))
order_items = pd.DataFrame(items, columns=["order_id", "product_id", "quantity", "discount_pct"])

print(customers.shape, orders.shape, order_items.shape)
orders.head()

## 2. Jakość danych

In [ ]:
print("Brakujące wartości:\n", orders.isna().sum())
print("\nDuplikaty order_id:", orders["order_id"].duplicated().sum())
print("\nZamówienia bez pozycji:", (~orders["order_id"].isin(order_items["order_id"])).sum())
print("Rozkład statusów:\n", orders["status"].value_counts(normalize=True).round(3))

## 3. Sprzedaż netto i kategorie

Odpowiednik zapytań 1 i 2 z pliku SQL.

In [ ]:
sales = (
    orders[orders["status"] == "completed"]
    .merge(order_items, on="order_id")
    .merge(products, on="product_id")
)
sales["net_revenue"] = sales["quantity"] * sales["unit_price"] * (1 - sales["discount_pct"] / 100)
sales["gross_margin"] = sales["quantity"] * (sales["unit_price"] * (1 - sales["discount_pct"] / 100) - sales["unit_cost"])

monthly = sales.set_index("ordered_at").resample("MS")["net_revenue"].sum()
ax = monthly.plot(marker="o", title="Miesięczna sprzedaż netto (PLN, symulacja)")
ax.set_ylabel("Przychód")
plt.tight_layout(); plt.show()

by_cat = sales.groupby("category").agg(revenue=("net_revenue", "sum"), margin=("gross_margin", "sum"))
by_cat["margin_pct"] = (by_cat["margin"] / by_cat["revenue"] * 100).round(1)
by_cat["revenue_share_pct"] = (by_cat["revenue"] / by_cat["revenue"].sum() * 100).round(1)
by_cat.sort_values("revenue", ascending=False)

## 4. Model churn

Cechy liczymy z okna `[T-180d, T-90d)`, etykieta to brak zakupów w `[T-90d, T)`. Dzięki temu model nie widzi przyszłości.

In [ ]:
T = END
feat_start, feat_end = T - pd.Timedelta(days=180), T - pd.Timedelta(days=90)

win = sales[(sales["ordered_at"] >= feat_start) & (sales["ordered_at"] < feat_end)]
features = (
    win.groupby("customer_id")
    .agg(
        orders_90d=("order_id", "nunique"),
        gross_spend_90d=("net_revenue", "sum"),
        avg_discount_90d=("discount_pct", "mean"),
    )
)
channels = orders[(orders["ordered_at"] >= feat_start) & (orders["ordered_at"] < feat_end)]
features["channels_used"] = channels.groupby("customer_id")["channel"].nunique()

label_win = sales[(sales["ordered_at"] >= feat_end) & (sales["ordered_at"] < T)]
active_after = set(label_win["customer_id"].unique())

data = customers.set_index("customer_id")[["segment", "city", "created_at"]].join(features, how="left").fillna(
    {"orders_90d": 0, "gross_spend_90d": 0, "avg_discount_90d": 0, "channels_used": 0}
)
data["churned_next_90d"] = (~data.index.isin(active_after)).astype(int)
data = data[data["created_at"] < feat_end]  # tylko klienci istniejący przed oknem cech

print("Odsetek churn:", data["churned_next_90d"].mean().round(3))
data.head()

In [ ]:
X = pd.get_dummies(data[["orders_90d", "gross_spend_90d", "avg_discount_90d", "channels_used", "segment", "city"]].astype({"channels_used": float}),
                   columns=["segment", "city"], drop_first=True).astype(float)
y = data["churned_next_90d"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = (model, proba)
    print(f"{name}: ROC AUC = {roc_auc_score(y_test, proba):.3f}")

best_name = max(results, key=lambda k: roc_auc_score(y_test, results[k][1]))
print("\nNajlepszy model:", best_name)
print(classification_report(y_test, results[best_name][1] >= 0.5))

In [ ]:
fig, ax = plt.subplots()
for name, (_, proba) in results.items():
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("Krzywa ROC – churn w ciągu 90 dni")
plt.tight_layout(); plt.show()

In [ ]:
model = results[best_name][0]
if hasattr(model, "feature_importances_"):
    importance = pd.Series(model.feature_importances_, index=X.columns)
else:
    importance = pd.Series(np.abs(model[-1].coef_[0]), index=X.columns)
importance.sort_values().tail(10).plot.barh(title="Najważniejsze cechy")
plt.tight_layout(); plt.show()

## 5. Wnioski i następne kroki

- Klienci z niskim wydatkiem i małą liczbą zamówień w ostatnim kwartale częściej odchodzą – kandydaci do kampanii reaktywacyjnej.
- Model należy walidować czasowo (np. kilka dat odniesienia), a nie tylko losowo.
- Przed wdrożeniem: monitoring driftu cech i kalibracja progu decyzyjnego pod koszt kampanii.